In [188]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge,LogisticRegression,SGDRegressor
from sklearn.metrics import accuracy_score, r2_score,mean_squared_error

In [189]:
X, y, coef = make_regression(n_samples=200,     # number of rows
                             n_features=3,       # number of input features
                             noise=10.0,         # add Gaussian noise
                             coef=True,          # return the true coefficients
                             random_state=42)

In [190]:
X = pd.DataFrame(X, columns=['x1', 'x2', 'x3'])
y = pd.Series(y, name='target')

In [192]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=2)

In [281]:
X_train.insert(0,0,1)

In [282]:
np.dot(np.linalg.inv(np.dot(X_train.values.T,X_train.values)),np.dot(X_train.values.T,y_train.values))

array([-0.45765889, 71.2116582 , 21.95232503, 73.06396211])

In [284]:
np.dot(X_train.values.T,X_train.values)

array([[160.        ,   0.52643461,   5.91714962, -25.88450099],
       [  0.52643461, 167.97274133, -11.01608882, -15.56530702],
       [  5.91714962, -11.01608882, 129.49675031,   8.43944501],
       [-25.88450099, -15.56530702,   8.43944501, 177.3253494 ]])

In [287]:
# ridge
np.dot(np.linalg.inv(np.dot(X_train.values.T,X_train.values) + 0.9*np.identity(4)),np.dot(X_train.values.T,y_train.values))

array([-0.51331161, 70.78445802, 21.79357551, 72.65712968])

In [288]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=2)

In [196]:
m = LinearRegression()
m.fit(X_train,y_train)
print(mean_squared_error(y_test,m.predict(X_test)))
print(r2_score(y_test,m.predict(X_test)))
print(m.intercept_,m.coef_)

125.83282519422667
0.984716480693054
-0.45765888503864005 [71.2116582  21.95232503 73.06396211]


In [ ]:
beta_not = 1 # initializing beta not
betas = np.ones(X_train.shape[1]) # initializing betas
lr = 0.001 # learning rate
epoch = 149
for i in range(epoch):
    cost = []
    shuffled_indices = np.random.permutation(X_train.shape[0]) # shuffling data to avoid biased update due to same order

    for j in shuffled_indices:
        y_hat = beta_not + np.dot(betas, X_train.values[j]) # calculating y_hat
        gradient_beta_not = -2*(y_train.values[j] - y_hat)  # calculating gradient for beta_not
        beta_not = beta_not - lr * gradient_beta_not # updating beta_not
        gradient_betas = -2 * (y_train.values[j] - y_hat) * X_train.values[j] # calculating betas gradient
        betas = betas - lr * gradient_betas # updating the betas
                
        loss = (y_train.values[j] - y_hat)**2 # calulating loss for every point
        cost.append(loss)
    print(f"Epoch {i+1} -> beta_not: {beta_not}, betas: {betas}, Loss: {np.mean(cost)}")

Epoch 1 -> beta_not: -1.9250226255180276, betas: [19.15638884  5.58095064 21.29070863], Loss: 7826.240216755934
Epoch 2 -> beta_not: -3.2149645192051426, betas: [32.69111408  9.20719322 35.71032627], Loss: 4213.997066445484
Epoch 3 -> beta_not: -3.7352663741364074, betas: [42.66415858 12.0165397  46.1304336 ], Loss: 2305.537883686172
Epoch 4 -> beta_not: -3.7428405948797683, betas: [50.10695975 14.20118745 53.56778689], Loss: 1287.688430712888
Epoch 5 -> beta_not: -3.511723009715377, betas: [55.59771152 15.89633239 58.91200708], Loss: 744.6236846413306
Epoch 6 -> beta_not: -3.1427906228719165, betas: [59.6515068  17.21657902 62.80606123], Loss: 452.8199132786981
Epoch 7 -> beta_not: -2.7582284922095397, betas: [62.65014641 18.24994635 65.59113934], Loss: 295.1043774753205
Epoch 8 -> beta_not: -2.3905246089205905, betas: [64.85767066 19.06900508 67.57045684], Loss: 209.97443167791303
Epoch 9 -> beta_not: -2.0127269053531736, betas: [66.49135154 19.70584001 68.98517888], Loss: 163.529012

In [209]:
prediction = []
for i in range(X_test.shape[0]):
    pred = beta_not + betas[0]*X_test.values[i][0] + betas[1]*X_test.values[i][1] + betas[2]*X_test.values[i][2] 
    prediction.append(pred)
print(mean_squared_error(y_test,prediction))
print(r2_score(y_test,prediction))

125.62367401931142
0.9847418839693053


In [224]:
# batch gradinet descent
beta_not = 1
betas = np.ones(X_train.shape[1])
lr = 0.1
epoch = 100
for i in range(epoch):
        
    y_hat = beta_not + X_train.values @ betas # @ is used for matrix multiplication
    error = y_train.values - y_hat # calculating error
    N = X_train.shape[0] # no. of rows
    gradient_beta_not = (-2*np.sum(error))/N # gradient for betanot
    beta_not = beta_not - lr * gradient_beta_not # updating beta not

    
    for j in range(len(betas)): # itrating on betas
        gradient = (-2*(np.sum(error*X_train.iloc[:,j])))/N  # calculating gradient for beta
        betas[j] = betas[j] - lr*gradient # updating the repective beta
        
    cost = np.mean((error)**2) # cal lo
    print(f"Epoch {i+1} -> Cost: {cost:.4f}, beta_not: {beta_not}, betas: {betas}")

Epoch 1 -> Cost: 10396.3030, beta_not: -1.42203153902208, betas: [14.05045871  4.1741935  15.87557252]
Epoch 2 -> Cost: 6731.3662, beta_not: -2.9104137946405375, betas: [24.69549713  6.87527102 27.59594102]
Epoch 3 -> Cost: 4389.0897, beta_not: -3.7488829819485066, betas: [33.37165267  9.17307292 36.84887606]
Epoch 4 -> Cost: 2885.4241, beta_not: -4.142978656891477, betas: [40.43833488 11.12698873 44.16827554]
Epoch 5 -> Cost: 1916.0251, beta_not: -4.240533640779286, betas: [46.19083045 12.78763168 49.96941137]
Epoch 6 -> Cost: 1288.5631, beta_not: -4.146946477342323, betas: [50.87129997 14.19820089 54.57593576]
Epoch 7 -> Cost: 880.8997, beta_not: -3.936542902873027, betas: [54.67801993 15.39560268 58.24060695]
Epoch 8 -> Cost: 615.1073, beta_not: -3.6610088055224277, betas: [57.77311048 16.41138291 61.1612212 ]
Epoch 9 -> Cost: 441.2431, beta_not: -3.355633082519227, betas: [60.28896889 17.27250915 63.49288111]
Epoch 10 -> Cost: 327.1633, beta_not: -3.043914997019329, betas: [62.3336

In [225]:
prediction = []
for i in range(X_test.shape[0]):
    pred = beta_not + betas[0]*X_test.values[i][0] + betas[1]*X_test.values[i][1] + betas[2]*X_test.values[i][2] 
    prediction.append(pred)
print(mean_squared_error(y_test,prediction))
print(r2_score(y_test,prediction))

125.83282545161451
0.984716480661792


In [289]:
m = Ridge(alpha=0.9)
m.fit(X_train,y_train)
print(mean_squared_error(y_test,m.predict(X_test)))
print(r2_score(y_test,m.predict(X_test)))
print(m.intercept_,m.coef_)

126.55131181366801
0.9846292140827473
-0.5162760667866113 [70.78443722 21.79373675 72.65668969]


In [ ]:
# SGD Ridge
beta_not = 1 # init beta not
betas = np.ones(X_train.shape[1]) # init betas
lr = 0.001 # learning rate
epoch = 130 # no. of epochs
lmda = 0.1 # lamda/alpha
for i in range(epoch):
    cost = []
    shuffled_indices = np.random.permutation(X_train.shape[0]) # shuffling data to avoid biased update due to same order
    for j in shuffled_indices:
        y_hat = beta_not + np.dot(betas, X_train.values[j])# calculating y_hat
        error = y_train.values[j] - y_hat # cal error
        gradient_beta_not = -2*error # gradient for beta_not
        beta_not = beta_not - lr * gradient_beta_not # updating beta_not
        for p in range(len(betas)): # iterating on betas
            gradient = (-2*error*X_train.values[j][p]) + (2*lmda*betas[p]) # cal gradient for beta
            betas[p] = betas[p] - lr*gradient # updating respective beta
        
        loss = (y_train.values[j] - y_hat)**2 # cal loss
        cost.append(loss)
    print(f"Epoch {i+1} -> beta_not: {beta_not}, betas: {betas}, Loss: {np.mean(cost)}")

Epoch 1 -> beta_not: -1.9647860256925909, betas: [18.89210312  5.49071349 21.0031884 ], Loss: 7848.201794348512
Epoch 2 -> beta_not: -3.412817079156455, betas: [31.6657121   8.91144027 34.8277802 ], Loss: 4313.728262262268
Epoch 3 -> beta_not: -3.9574883544760486, betas: [40.91437787 11.46807576 44.30370291], Loss: 2479.039113974473
Epoch 4 -> beta_not: -4.026701691262849, betas: [47.48262216 13.4059491  50.95446017], Loss: 1506.7824467494233
Epoch 5 -> beta_not: -3.837937222155702, betas: [52.20601042 14.88146468 55.61986763], Loss: 973.1512308240951
Epoch 6 -> beta_not: -3.5386339511013967, betas: [55.60206275 15.98796362 58.83800066], Loss: 671.3047693517108
Epoch 7 -> beta_not: -3.212400661324607, betas: [58.07170585 16.80127944 61.02970755], Loss: 493.840547553635
Epoch 8 -> beta_not: -2.8672318650639124, betas: [59.88524482 17.41664624 62.63912312], Loss: 397.8591909204764
Epoch 9 -> beta_not: -2.5446532564039357, betas: [61.15383279 17.93175952 63.78073374], Loss: 328.1398859428

In [ ]:
# batch GD rigde
beta_not = 1# init beta not
betas = np.ones(X_train.shape[1])# init betas
lr = 0.03 # learning rate
epoch = 150# no. of epochs
lmda = 0# lamda/alpha
for i in range(epoch):
    y_hat = beta_not + X_train.values @ betas # cal y_hat @ is used for matrix multiplication
    error = y_train.values - y_hat # error
    N = X_train.shape[0] # no. of rows
    gradient_beta_not = (-2*np.sum(error))/N  # gradient for beta not
    beta_not = beta_not - lr * gradient_beta_not # updating beta not
    
    for j in range(len(betas)): # itrating on betas
        gradient = ((-2*(np.sum(error*X_train.iloc[:,j])))/N) + (2*lmda*betas[j]) # cal gradient for beta
        betas[j] = betas[j] - lr*gradient # updating betas
        
    Cost = np.mean((error)**2) # cal cost
    print(f"Epoch {i+1} -> Cost: {Cost:.4f}, beta_not: {beta_not}, betas: {betas}")

Epoch 1 -> Cost: 10396.3030, beta_not: 0.2733905382933761, betas: [4.91513761 1.95225805 5.46267175]
Epoch 2 -> Cost: 9210.6270, beta_not: -0.36919048790692166, betas: [8.6137874  2.86193566 9.64137515]
Epoch 3 -> Cost: 8163.7643, beta_not: -0.9354039564084081, betas: [12.10773587  3.73091853 13.55467999]
Epoch 4 -> Cost: 7239.1869, beta_not: -1.432277352079502, betas: [15.4081448   4.56100768 17.2199097 ]
Epoch 5 -> Cost: 6422.3689, beta_not: -1.8662545573660898, betas: [18.52558283  5.35392358 20.65322702]
Epoch 6 -> Cost: 5700.5403, beta_not: -2.243241839274316, betas: [21.4700556   6.11130993 23.86971357]
Epoch 7 -> Cost: 5062.4705, beta_not: -2.5686503185468896, betas: [24.2510345   6.83473735 26.88344399]
Epoch 8 -> Cost: 4498.2797, beta_not: -2.8474351855022726, betas: [26.87748401  7.52570675 29.70755477]
Epoch 9 -> Cost: 3999.2735, beta_not: -3.084131907321579, betas: [29.35788781  8.18565263 32.35430834]
Epoch 10 -> Cost: 3557.7983, beta_not: -3.282889653340698, betas: [31.70

In [316]:
m = SGDRegressor(alpha=0.5,penalty='l1')
m.fit(X_train,y_train)
print(mean_squared_error(y_test,m.predict(X_test)))
print(r2_score(y_test,m.predict(X_test)))
print(m.intercept_,m.coef_)

126.73853535441461
0.9846064741133053
[-0.5117035] [70.66068833 21.32207647 72.57756353]


In [ ]:
# SGD lasso
beta_not = 1 # init beta not
betas = np.ones(X_train.shape[1]) # init betas
lr = 0.001 # learning rate
epoch = 130 # no. of epochs
lmda = 0.5 # lamda/alpha
for i in range(epoch):
    cost = []
    shuffled_indices = np.random.permutation(X_train.shape[0]) # shuffling data to avoid biased update due to same order
    for j in shuffled_indices:
        y_hat = beta_not + np.dot(betas, X_train.values[j])# calculating y_hat
        error = y_train.values[j] - y_hat # cal error
        gradient_beta_not = -2*error # gradient for beta_not
        beta_not = beta_not - lr * gradient_beta_not # updating beta_not
        for p in range(len(betas)): # iterating on betas
            gradient = (-2*error*X_train.values[j][p]) + lmda*np.sign(betas[p]) # cal gradient for beta
            betas[p] = betas[p] - lr*gradient # updating respective beta
        
        loss = (y_train.values[j] - y_hat)**2 # cal loss
        cost.append(loss)
    print(f"Epoch {i+1} -> beta_not: {beta_not}, betas: {betas}, Loss: {np.mean(cost)}")

Epoch 1 -> beta_not: -1.7662721187372654, betas: [19.16034671  5.50943916 21.16298603], Loss: 7836.7841497830495
Epoch 2 -> beta_not: -3.2148289935241428, betas: [32.5196771   9.01587087 35.69642426], Loss: 4232.477867839571
Epoch 3 -> beta_not: -3.6811697186058776, betas: [42.45088557 11.79033415 46.047937  ], Loss: 2323.8217656422808
Epoch 4 -> beta_not: -3.71543655989348, betas: [49.86954855 13.92987115 53.43725365], Loss: 1307.520946571683
Epoch 5 -> beta_not: -3.4850352675958254, betas: [55.35365891 15.6359847  58.75389522], Loss: 761.3228958792611
Epoch 6 -> beta_not: -3.0949296558253026, betas: [59.41280579 16.9598449  62.53968026], Loss: 466.371881645471
Epoch 7 -> beta_not: -2.7053088059257524, betas: [62.39711116 17.98798217 65.29772821], Loss: 306.5908021876491
Epoch 8 -> beta_not: -2.33863429435091, betas: [64.63378894 18.81317051 67.32188637], Loss: 218.0986153749057
Epoch 9 -> beta_not: -1.9547147907115385, betas: [66.25598908 19.45249327 68.8205328 ], Loss: 168.995057272

In [318]:
# batch GD lasso
beta_not = 1# init beta not
betas = np.ones(X_train.shape[1])# init betas
lr = 0.03 # learning rate
epoch = 150# no. of epochs
lmda = 0.5# lamda/alpha
for i in range(epoch):
    y_hat = beta_not + X_train.values @ betas # cal y_hat @ is used for matrix multiplication
    error = y_train.values - y_hat # error
    N = X_train.shape[0] # no. of rows
    gradient_beta_not = (-2*np.sum(error))/N  # gradient for beta not
    beta_not = beta_not - lr * gradient_beta_not # updating beta not
    
    for j in range(len(betas)): # itrating on betas
        gradient = ((-2*(np.sum(error*X_train.iloc[:,j])))/N) + lmda*np.sign(betas[j]) # cal gradient for beta
        betas[j] = betas[j] - lr*gradient # updating betas
        
    Cost = np.mean((error)**2) # cal cost
    print(f"Epoch {i+1} -> Cost: {Cost:.4f}, beta_not: {beta_not}, betas: {betas}")

Epoch 1 -> Cost: 10396.3030, beta_not: 0.2733905382933761, betas: [4.90013761 1.93725805 5.44767175]
Epoch 2 -> Cost: 9215.0211, beta_not: -0.3692998430636978, betas: [8.58458273 2.83264959 9.61233252]
Epoch 3 -> Cost: 8171.8027, beta_not: -0.935717908816476, betas: [12.06508031  3.68802613 13.51248976]
Epoch 4 -> Cost: 7250.2207, beta_not: -1.4328783986561933, betas: [15.35275271  4.5051562  17.16540872]
Epoch 5 -> Cost: 6435.8381, beta_not: -1.867213699484466, betas: [18.45813115  5.28572931 20.58719779]
Epoch 6 -> Cost: 5715.9619, beta_not: -2.244619723775451, betas: [21.39118575  6.03135972 23.79288788]
Epoch 7 -> Cost: 5079.4292, beta_not: -2.5704982790053514, betas: [24.16135421  6.74359    26.79650627]
Epoch 8 -> Cost: 4516.4190, beta_not: -2.8497961919192814, betas: [26.77756904  7.42389441 29.61114521]
Epoch 9 -> Cost: 4018.2882, beta_not: -3.08704143177488, betas: [29.2482836   8.07368202 32.24902582]
Epoch 10 -> Cost: 3577.4280, beta_not: -3.2863764585245776, betas: [31.5814